# GRPO Training Notebook for ITSM OpenEnv

This notebook trains a policy model with GRPO against your deployed environment:
- https://rahulsamant37-itsm-openenv-benchmark.hf.space

It includes:
1. TRL GRPO setup
2. 4-bit quantization (bitsandbytes)
3. LoRA adapters (QLoRA style)
4. Best-effort TorchForge integration with safe fallback
5. Environment-backed reward function
6. Quick post-training evaluation

## Model Choice

Chosen free model: **Qwen/Qwen2.5-1.5B-Instruct**

Why this model:
- Open and free to use
- Strong instruction-following for structured JSON action outputs
- Small enough to train with 4-bit quantization on budget GPUs

In [12]:
import sys
import shutil
import subprocess

base_deps = [
    "trl>=0.13.0",
    "transformers>=4.45.0",
    "accelerate>=0.34.0",
    "datasets>=2.20.0",
    "peft>=0.12.0",
    "huggingface_hub>=0.24.0",
    "sentencepiece",
    "requests",
    "numpy",
]

optional_deps = [
    "bitsandbytes>=0.43.0",
    "torchforge",
]

# Some uv-managed venvs do not ship pip by default. Try to bootstrap it first.
try:
    import ensurepip
    ensurepip.bootstrap(upgrade=True)
except Exception:
    pass

def run_install(packages: list[str], required: bool = True) -> bool:
    pip_cmd = [sys.executable, "-m", "pip", "install", "-U"] + packages
    uv_cmd = ["uv", "pip", "install", "-p", sys.executable, "-U"] + packages

    attempted = []

    for cmd in (pip_cmd, uv_cmd):
        if cmd[0] == "uv" and shutil.which("uv") is None:
            continue
        attempted.append(" ".join(cmd))
        result = subprocess.run(cmd)
        if result.returncode == 0:
            return True

    if required:
        raise RuntimeError("Failed to install required dependencies. Tried:\n" + "\n".join(attempted))
    return False

print("Installing required dependencies...")
run_install(base_deps, required=True)

print("Attempting optional dependency: bitsandbytes")
if run_install([optional_deps[0]], required=False):
    print("bitsandbytes installed.")
else:
    print("bitsandbytes unavailable; notebook will fallback to non-4bit path if needed.")

print("Attempting optional dependency: torchforge")
if run_install([optional_deps[1]], required=False):
    print("TorchForge installation succeeded.")
else:
    print("TorchForge install unavailable in this runtime. Fallback to native TRL.")

Looking in links: /tmp/tmpfrhwg1xx
Installing required dependencies...



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


Attempting optional dependency: bitsandbytes



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


bitsandbytes installed.
Attempting optional dependency: torchforge
TorchForge installation succeeded.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [26]:
import json
import math
import os
import random
import re
import warnings
from typing import Any

import numpy as np
import requests
import torch
from datasets import Dataset
from huggingface_hub import notebook_login
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer

warnings.filterwarnings(
    "ignore",
    message=r".*_check_is_size will be removed in a future PyTorch release.*",
    category=FutureWarning,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

SPACE_URL = os.getenv("ITSM_SPACE_URL", "https://rahulsamant37-itsm-openenv-benchmark.hf.space").rstrip("/")
MODEL_NAME = os.getenv("MODEL_NAME", "Qwen/Qwen2.5-1.5B-Instruct")
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "./outputs/itsm-grpo-qwen2.5-1.5b")
RAW_TASKS_URL = os.getenv("RAW_TASKS_URL", "https://raw.githubusercontent.com/rahulsamant37/meta-hackathon/main/tasks.jsonl")

# Defaults tuned for practical notebook runtime; override via env if you want longer runs.
TRAIN_TASK_LIMIT = int(os.getenv("TRAIN_TASK_LIMIT", "16"))
REPEAT_FACTOR = int(os.getenv("REPEAT_FACTOR", "1"))
EVAL_TASKS = int(os.getenv("EVAL_TASKS", "8"))
NUM_GENERATIONS = int(os.getenv("NUM_GENERATIONS", "2"))
TRAIN_MAX_STEPS = int(os.getenv("TRAIN_MAX_STEPS", "5"))

MAX_PROMPT_LENGTH = int(os.getenv("MAX_PROMPT_LENGTH", "512"))
MAX_COMPLETION_LENGTH = int(os.getenv("MAX_COMPLETION_LENGTH", "96"))

print("Config loaded:")
print(json.dumps({
    "SPACE_URL": SPACE_URL,
    "MODEL_NAME": MODEL_NAME,
    "OUTPUT_DIR": OUTPUT_DIR,
    "TRAIN_TASK_LIMIT": TRAIN_TASK_LIMIT,
    "REPEAT_FACTOR": REPEAT_FACTOR,
    "EVAL_TASKS": EVAL_TASKS,
    "NUM_GENERATIONS": NUM_GENERATIONS,
    "TRAIN_MAX_STEPS": TRAIN_MAX_STEPS,
    "MAX_PROMPT_LENGTH": MAX_PROMPT_LENGTH,
    "MAX_COMPLETION_LENGTH": MAX_COMPLETION_LENGTH,
}, indent=2))

Config loaded:
{
  "SPACE_URL": "https://rahulsamant37-itsm-openenv-benchmark.hf.space",
  "MODEL_NAME": "Qwen/Qwen2.5-1.5B-Instruct",
  "OUTPUT_DIR": "./outputs/itsm-grpo-qwen2.5-1.5b",
  "TRAIN_TASK_LIMIT": 16,
  "REPEAT_FACTOR": 1,
  "EVAL_TASKS": 8,
  "NUM_GENERATIONS": 2,
  "TRAIN_MAX_STEPS": 5,
  "MAX_PROMPT_LENGTH": 512,
  "MAX_COMPLETION_LENGTH": 96
}


In [14]:
session = requests.Session()

health = session.get(f"{SPACE_URL}/health", timeout=30)
health.raise_for_status()
print("Environment health:", health.json())

def load_tasks() -> list[dict[str, Any]]:
    local_path = "tasks.jsonl"
    tasks: list[dict[str, Any]] = []

    if os.path.exists(local_path):
        with open(local_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    tasks.append(json.loads(line))
        return tasks

    resp = session.get(RAW_TASKS_URL, timeout=60)
    resp.raise_for_status()
    for line in resp.text.splitlines():
        line = line.strip()
        if line:
            tasks.append(json.loads(line))
    return tasks

tasks = load_tasks()
print(f"Loaded tasks: {len(tasks)}")
print("Task types:", sorted({t.get("source_table", "") for t in tasks}))

Environment health: {'status': 'healthy'}
Loaded tasks: 181
Task types: ['incident', 'incident_knowledge', 'incident_sla', 'problem']


## Prompt + Action Formatting

The model is trained to output **one JSON action object** only.

In [15]:
ACTION_TYPES = [
    "query",
    "update_incident",
    "update_problem",
    "update_sla",
    "attach_knowledge",
    "add_worknote",
    "set_assignment",
    "resolve",
    "close",
    "noop",
]

SYSTEM_PROMPT = (
    "You are an ITSM agent. Produce exactly one valid JSON object for the next action. "
    "No markdown, no prose, no code fences."
)

def build_prompt(task: dict[str, Any]) -> str:
    allowed = task.get("allowed_action_types", ACTION_TYPES)
    objective = task.get("objective", "")
    task_id = task.get("task_id", "UNKNOWN")
    task_type = task.get("source_table", "incident")

    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"Task ID: {task_id}\n"
        f"Task Type: {task_type}\n"
        f"Objective: {objective}\n"
        f"Allowed action_type values for this task: {allowed}\n\n"
        "Return JSON with keys: action_type, target_id, payload, reasoning.\n"
        "Example: {\"action_type\": \"query\", \"target_id\": null, \"payload\": {}, \"reasoning\": \"inspect current state\"}"
    )

def extract_json_object(text: str) -> dict[str, Any] | None:
    text = text.strip()
    try:
        maybe = json.loads(text)
        if isinstance(maybe, dict):
            return maybe
    except Exception:
        pass

    match = re.search(r"\{[\s\S]*\}", text)
    if not match:
        return None

    snippet = match.group(0)
    try:
        maybe = json.loads(snippet)
        if isinstance(maybe, dict):
            return maybe
    except Exception:
        return None
    return None

def normalize_action(raw: dict[str, Any] | None, allowed_actions: list[str]) -> tuple[dict[str, Any], bool]:
    if raw is None:
        return {
            "action_type": "noop",
            "target_id": None,
            "payload": {},
            "reasoning": "invalid json fallback",
        }, False

    action_type = str(raw.get("action_type", "noop"))
    if action_type not in ACTION_TYPES:
        action_type = "noop"

    if allowed_actions and action_type not in allowed_actions:
        action_type = "noop"

    target_id = raw.get("target_id")
    payload = raw.get("payload", {})
    reasoning = raw.get("reasoning", "")

    if not isinstance(payload, dict):
        payload = {}

    action = {
        "action_type": action_type,
        "target_id": target_id if isinstance(target_id, str) else None,
        "payload": payload,
        "reasoning": str(reasoning)[:300] if reasoning is not None else None,
    }
    return action, True

In [16]:
def reset_task(task_id: str) -> dict[str, Any]:
    resp = session.post(f"{SPACE_URL}/reset", json={"task_id": task_id}, timeout=30)
    resp.raise_for_status()
    return resp.json()

def step_task(action: dict[str, Any]) -> dict[str, Any]:
    resp = session.post(f"{SPACE_URL}/step", json=action, timeout=30)
    resp.raise_for_status()
    return resp.json()

def to_open_interval(x: float, eps: float = 1e-3) -> float:
    if math.isnan(x) or math.isinf(x):
        return eps
    return min(1.0 - eps, max(eps, float(x)))

sample = tasks[0]
obs0 = reset_task(sample["task_id"])
print("Sample task_id:", sample["task_id"])
print("Allowed actions:", obs0.get("allowed_actions", []))
print("Objective:", obs0.get("progress_signals", {}).get("objective", ""))

Sample task_id: ITSM-001
Allowed actions: ['query', 'update_incident', 'set_assignment', 'add_worknote', 'resolve', 'close', 'noop']
Objective: Triage and progress incident INC-000001 (MAJOR:SAP Database Unreachable) to a correct lifecycle state.


## GRPO Reward Function (Environment-backed)

For each generated completion:
1. Reset the corresponding task
2. Parse model output into an action
3. Execute one step in the environment
4. Compute reward from format validity + action validity + progress

In [17]:
def completion_to_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion

    if isinstance(completion, dict):
        if "content" in completion and isinstance(completion["content"], str):
            return completion["content"]
        return json.dumps(completion, ensure_ascii=False)

    if isinstance(completion, list):
        parts = []
        for item in completion:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and isinstance(item.get("content"), str):
                parts.append(item["content"])
            else:
                parts.append(str(item))
        return "\n".join(parts)

    return str(completion)

def itsm_env_reward(completions: list[Any], **kwargs) -> list[float]:
    task_ids = kwargs.get("task_id", None)
    if task_ids is None:
        task_ids = [tasks[0]["task_id"]] * len(completions)

    rewards: list[float] = []

    for completion, task_id in zip(completions, task_ids):
        text = completion_to_text(completion)

        try:
            obs = reset_task(str(task_id))
            allowed = obs.get("allowed_actions", []) or []

            raw_action = extract_json_object(text)
            action, format_ok = normalize_action(raw_action, allowed)

            result = step_task(action)

            normalized_progress = float(result.get("reward", {}).get("normalized_progress", 0.0))
            grader_score = float(result.get("info", {}).get("grader_score", normalized_progress))

            action_error = (result.get("observation", {}) or {}).get("last_action_error")
            valid_action = 1.0 if not action_error else 0.0
            format_score = 1.0 if format_ok else 0.0

            reward = (0.20 * format_score) + (0.15 * valid_action) + (0.65 * grader_score)
            rewards.append(to_open_interval(reward))
        except Exception:
            rewards.append(1e-3)

    while len(rewards) < len(completions):
        rewards.append(1e-3)

    return rewards

In [18]:
selected_tasks = tasks[: min(TRAIN_TASK_LIMIT, len(tasks))]
records: list[dict[str, Any]] = []

for _ in range(REPEAT_FACTOR):
    random.shuffle(selected_tasks)
    for task in selected_tasks:
        records.append({
            "prompt": build_prompt(task),
            "task_id": task["task_id"],
        })

train_dataset = Dataset.from_list(records)
print("Training rows:", len(train_dataset))
print("First task_id:", train_dataset[0]["task_id"])
print("Prompt preview:")
print(train_dataset[0]["prompt"][:500])

Training rows: 16
First task_id: ITSM-008
Prompt preview:
You are an ITSM agent. Produce exactly one valid JSON object for the next action. No markdown, no prose, no code fences.

Task ID: ITSM-008
Task Type: incident
Objective: Triage and progress incident INC-000005 (Request New Software) to a correct lifecycle state.
Allowed action_type values for this task: ['query', 'update_incident', 'set_assignment', 'add_worknote', 'resolve', 'close', 'noop']

Return JSON with keys: action_type, target_id, payload, reasoning.
Example: {"action_type": "query", "


In [19]:
compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

use_4bit = torch.cuda.is_available()
if use_4bit:
    try:
        import bitsandbytes  # noqa: F401
    except Exception as exc:
        use_4bit = False
        print(f"bitsandbytes unavailable, falling back to non-4bit load: {exc}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if use_4bit:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    print("Model loaded with 4-bit quantization.")
else:
    fallback_dtype = (
        torch.bfloat16
        if (torch.cuda.is_available() and torch.cuda.is_bf16_supported())
        else (torch.float16 if torch.cuda.is_available() else torch.float32)
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=fallback_dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )
    print("Model loaded without 4-bit quantization.")

model.config.use_cache = False
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Loading weights: 100%|██████████| 338/338 [00:02<00:00, 126.88it/s]


Model loaded with 4-bit quantization.
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [20]:
TORCHFORGE_AVAILABLE = False
torchforge_msg = "TorchForge not imported"

try:
    import torchforge  # type: ignore
    TORCHFORGE_AVAILABLE = True
    torchforge_msg = f"TorchForge detected: version={getattr(torchforge, '__version__', 'unknown')}"

    for fn_name in ["prepare_model", "optimize_model", "optimize", "prepare"]:
        fn = getattr(torchforge, fn_name, None)
        if callable(fn):
            try:
                maybe_model = fn(model)
                if maybe_model is not None:
                    model = maybe_model
                torchforge_msg += f" | applied hook: {fn_name}"
                break
            except Exception as hook_exc:
                torchforge_msg += f" | hook {fn_name} failed: {hook_exc}"
except Exception as exc:
    torchforge_msg = f"TorchForge unavailable, using native TRL path: {exc}"

print(torchforge_msg)

TorchForge detected: version=0.0.1


In [22]:
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

grpo_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    max_steps=TRAIN_MAX_STEPS,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    logging_steps=1,
    save_steps=25,
    remove_unused_columns=False,
    report_to="none",
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=torch.cuda.is_available(),
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[itsm_env_reward],
    train_dataset=train_dataset,
    args=grpo_args,
    peft_config=peft_config,
)

print("GRPO trainer initialized.")

GRPO trainer initialized.


In [23]:
train_result = trainer.train()
print("Training finished.")
print(train_result)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,0.000000
2,0.000000
3,-0.000000
4,0.000000
5,0.000000


Training finished.
TrainOutput(global_step=5, training_loss=7.152557373046875e-08, metrics={'train_runtime': 48.9773, 'train_samples_per_second': 0.408, 'train_steps_per_second': 0.102, 'total_flos': 0.0, 'train_loss': 7.152557373046875e-08})


In [24]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved model artifacts to: {OUTPUT_DIR}")

if os.getenv("HF_PUSH_REPO"):
    repo_id = os.environ["HF_PUSH_REPO"]
    print(f"Pushing to Hub repo: {repo_id}")
    notebook_login()
    trainer.push_to_hub(repo_id=repo_id)
    tokenizer.push_to_hub(repo_id)

Saved model artifacts to: ./outputs/itsm-grpo-qwen2.5-1.5b


## Quick Evaluation (One-Step Policy Check)

This evaluates the trained model on one action per task for a small sample.

In [27]:
@torch.no_grad()
def generate_action_text(prompt: str, max_new_tokens: int = 160) -> str:
    # Eval generation should use cache and disable gradient checkpointing to avoid warnings.
    if hasattr(model, "gradient_checkpointing_disable"):
        try:
            model.gradient_checkpointing_disable()
        except Exception:
            pass

    if hasattr(model, "config"):
        model.config.use_cache = True

    if getattr(model, "generation_config", None) is not None:
        model.generation_config.do_sample = False
        model.generation_config.temperature = None
        model.generation_config.top_p = None
        model.generation_config.top_k = None

    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    gen = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True)

eval_pool = tasks[-EVAL_TASKS:] if EVAL_TASKS < len(tasks) else tasks
scores = []

for task in eval_pool:
    task_id = task["task_id"]
    obs = reset_task(task_id)

    prompt = build_prompt(task)
    text = generate_action_text(prompt)
    action, _ = normalize_action(extract_json_object(text), obs.get("allowed_actions", []))

    result = step_task(action)
    s = float(result.get("info", {}).get("grader_score", 0.0))
    scores.append(s)

if scores:
    arr = np.array(scores, dtype=np.float32)
    print({
        "eval_tasks": int(arr.shape[0]),
        "mean_score": float(arr.mean()),
        "median_score": float(np.median(arr)),
        "min_score": float(arr.min()),
        "max_score": float(arr.max()),
    })
else:
    print("No scores collected.")

{'eval_tasks': 8, 'mean_score': 0.8999999761581421, 'median_score': 0.8999999761581421, 'min_score': 0.8999999761581421, 'max_score': 0.8999999761581421}


## Notes

- This notebook uses one-step environment reward per sampled task prompt for GRPO.
- For deeper policies, extend reward rollout to multi-step episodes per completion.
- If TorchForge is unavailable in the runtime, training still proceeds with native TRL + QLoRA.
- For low-memory GPUs, reduce TRAIN_TASK_LIMIT, REPEAT_FACTOR, and num_generations.